# Day 1 · Lab B — Neural Networks in PyTorch
**Goal:** build an MLP, train it with the canonical loop (**forward → loss → backprop → step**),
read the loss curves, and see **regularization** fight overfitting.

**Domain:** predict customer **churn** (same data as Lab A).

In [ ]:
"""
PyTorch is a library for building neural networks (brain-like computer models).
A 'tensor' is just a fancy array (like a spreadsheet) that can remember how it was calculated,
so we can use math to automatically figure out what to change to improve predictions.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# Set random seeds for reproducibility (so we get the same results every time)
torch.manual_seed(7); np.random.seed(7)
# Print the PyTorch version being used
print('torch', torch.__version__)

torch 2.3.1+cpu


## 1. Tensors & autograd in 60 seconds
A `tensor` is an n-D array that can track gradients. `.backward()` fills in `.grad`.

In [ ]:
"""
This demonstrates PyTorch's "magic" - it can automatically compute how much to change x
to minimize a formula. This is like having a calculator that not only solves y = x² + 2x,
but tells you "if you want to improve y, change x by this amount".
"""

# Create a variable x = 3, and tell PyTorch to track how this affects other variables
x = torch.tensor(3.0, requires_grad=True)
# Calculate y = x² + 2x (derivative: dy/dx = 2x + 2, so at x=3: dy/dx = 8)
y = x ** 2 + 2 * x
# Backward pass: PyTorch automatically calculates dy/dx and stores it in x.grad
y.backward()
# Print the gradient (how much y changes when x changes slightly)
print('autograd dy/dx at x=3:', x.grad.item(), '(expected 8)')

autograd dy/dx at x=3: 8.0 (expected 8)


## 2. Prepare data as tensors

In [ ]:
"""
We prepare the same customer churn data as Lab A, but now we convert it to PyTorch "tensors"
(special arrays that track calculations for learning). We also organize the data into
small batches so the neural network can learn efficiently (like studying with flashcards).
"""

# Find the data directory
DATA = next((p for p in ('datasets', '../datasets', '../../datasets') if os.path.isdir(p)), None)
if DATA is None:
    raise FileNotFoundError("Could not find a datasets directory. Tried: datasets, ../datasets, ../../datasets")

# Load the CSV file
df = pd.read_csv(os.path.join(DATA, 'telco_customer_churn.csv'))

# Clean data: convert TotalCharges to numbers, remove bad rows, drop ID column
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna().drop(columns=['customerID'])

# Convert target (Churn: Yes/No) to binary numbers (1/0) and prepare for PyTorch
y = (df['Churn'] == 'Yes').astype(np.float32).values
# Convert categorical features to binary columns (one-hot encoding)
X = pd.get_dummies(df.drop(columns=['Churn']), drop_first=True).astype(np.float32).values

# Split into training (80%) and test (20%) sets
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=7, stratify=y)

# Normalize features (subtract mean, divide by std) - helps neural networks learn faster
scaler = StandardScaler().fit(Xtr)
Xtr, Xte = scaler.transform(Xtr), scaler.transform(Xte)

# Convert numpy arrays to PyTorch tensors (special arrays that track calculations)
to_t = lambda a: torch.tensor(a, dtype=torch.float32)

# Create PyTorch datasets (organize data for training)
train_ds = TensorDataset(to_t(Xtr), to_t(ytr).unsqueeze(1))
test_ds = TensorDataset(to_t(Xte), to_t(yte).unsqueeze(1))

# Create DataLoader - breaks data into batches of 128 customers and shuffles them
# Shuffling helps the model learn better (like randomizing flashcard order)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)

# Print the number of input features (customer attributes) we're using
print('input features:', Xtr.shape[1])

input features: 30


## 3. Define the MLP
### Exercise 1 — finish the `forward` pass
Return the logits from `self.net(x)`.

In [ ]:
"""
MLP stands for "Multi-Layer Perceptron" - it's like a network of connected neurons.
Each layer processes information and passes it to the next layer, just like how your brain
passes signals through connected neurons.

Architecture:
- Input: customer features (e.g., age, tenure, contract type)
- Layer 1: 64 neurons that learn patterns → ReLU activation → Dropout (random forgetting to prevent overconfidence)
- Layer 2: 32 neurons that refine patterns → ReLU activation → Dropout
- Output: 1 neuron that predicts churn probability
"""

class MLP(nn.Module):
    # Initialize the network with d_in input features and dropout probability p_drop
    def __init__(self, d_in, p_drop=0.0):
        super().__init__()
        # Build the neural network layers
        self.net = nn.Sequential(
            # Layer 1: d_in features → 64 neurons
            nn.Linear(d_in, 64),
            # Activation: ReLU (Rectified Linear Unit) - keeps positive values, zeros out negative
            nn.ReLU(),
            # Dropout: randomly "turn off" p_drop fraction of neurons during training (prevents overfitting)
            nn.Dropout(p_drop),
            
            # Layer 2: 64 neurons → 32 neurons (compress learned patterns)
            nn.Linear(64, 32),
            # Activation: ReLU again
            nn.ReLU(),
            # Dropout: prevent overfitting
            nn.Dropout(p_drop),
            
            # Output layer: 32 neurons → 1 output (final churn prediction)
            nn.Linear(32, 1),
        )
    
    # Forward pass: pass input through all layers
    def forward(self, x):
        # EXERCISE 1: Complete this - return self.net applied to x
        # This runs the input through all the layers and returns the prediction
        return self.net(x)

## 4. The canonical training loop
### Exercise 2 — fill in the loop: `zero_grad`, then **forward → loss → backprop → step**.

In [ ]:
"""
This is the "canonical" (standard) training loop - the same 4-step process used for ALL neural networks:
  1. Forward: feed data through the network to get predictions
  2. Loss: measure how wrong the predictions are
  3. Backward: calculate how to change weights to reduce the loss (PyTorch's automatic differentiation)
  4. Step: update the weights to improve predictions

Weight decay is like a penalty for having very large weights (prevents the model from becoming too extreme).
"""

def train(model, weight_decay=0.0, epochs=40, lr=1e-3):
    # Loss function: Binary Cross-Entropy (measures how wrong the binary prediction is)
    # WithLogits means: takes raw predictions (before probability conversion)
    loss_fn = nn.BCEWithLogitsLoss()
    
    # Optimizer: AdamW updates the model weights based on gradients
    # lr=learning rate (how big a step to take), weight_decay=regularization strength
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    # Prepare test data as tensors (no gradients needed for evaluation)
    Xte_t, yte_t = test_ds.tensors
    
    # Track training and validation loss over epochs
    hist = {'train': [], 'val': []}
    
    # Training loop: repeat epochs times (each epoch = one pass through all training data)
    for _ in range(epochs):
        # Set model to training mode (enables dropout, batch normalization, etc.)
        model.train()
        running = 0.0
        
        # Loop through batches of data (128 customers at a time)
        for xb, yb in train_dl:
            # EXERCISE 2: Fill in the canonical loop
            # Step 1: Clear old gradients from previous batch
            opt.zero_grad()
            
            # Step 2: Forward pass - feed batch through network to get predictions
            logits = model(xb)
            
            # Step 3a: Calculate loss (how wrong are our predictions?)
            loss = loss_fn(logits, yb)
            
            # Step 3b: Backward pass - calculate gradients (how to improve?)
            loss.backward()
            
            # Step 4: Update weights using gradients
            opt.step()
            
            # Accumulate loss for tracking (multiply by batch size to get total loss)
            running += loss.item() * len(xb)
        
        # Evaluation mode: disable dropout and other training-specific behaviors
        model.eval()
        
        # Evaluate on test data (no gradient calculation needed)
        with torch.no_grad():
            # Calculate validation loss
            val = loss_fn(model(Xte_t), yte_t).item()
        
        # Store average training and validation loss for this epoch
        hist['train'].append(running / len(train_ds))
        hist['val'].append(val)
    
    # Return loss history so we can plot learning curves
    return hist

## 5. Train the baseline (no regularization)

In [ ]:
"""
Train a baseline neural network WITHOUT regularization (no dropout, no weight decay).
This will likely overfit - memorizing training data but failing on new customers.
We'll compare this to a regularized version later.
"""

# Create MLP model with input features (no dropout = no regularization)
model = MLP(Xtr.shape[1])

# Train for 40 epochs with no regularization (default weight_decay=0.0, p_drop=0.0)
hist = train(model)

# Evaluate on test data
Xte_t, yte_t = test_ds.tensors
with torch.no_grad():
    # Convert model predictions (logits) to probabilities using sigmoid
    # Calculate ROC-AUC score (measure how well the model ranks customers by churn risk)
    auc = roc_auc_score(yte_t.numpy(), torch.sigmoid(model(Xte_t)).numpy())

# Print the test ROC-AUC score
print('baseline test ROC-AUC: %.3f' % auc)

# Plot training and validation loss curves
# The gap between them shows if the model is overfitting (memorizing vs. learning)
plt.plot(hist['train'], label='train')
plt.plot(hist['val'], label='val')
plt.xlabel('epoch')
plt.ylabel('BCE loss')
plt.title('Baseline MLP')
plt.legend()
plt.grid(True)
plt.show()

AttributeError: 'ellipsis' object has no attribute 'item'

## 6. Regularize
### Exercise 3 — train with `p_drop=0.3` and `weight_decay=1e-3`, then compare curves.

In [ ]:
"""

Regularization is like "punishing the model for being overconfident".
- Dropout (p_drop=0.3): randomly turn off 30% of neurons during training
  (forces the network to learn robust features instead of memorizing)
- Weight decay (1e-3): penalize large weights
  (prevents the network from becoming too extreme or sensitive to small changes)

We'll train this regularized version and compare loss curves to the baseline.
Expect: higher training loss, but better validation loss (less overfitting).
"""

# Train a regularized MLP with dropout and weight decay
# p_drop=0.3 means 30% of neurons are randomly disabled during training
reg_model = MLP(Xtr.shape[1], p_drop=0.3)

# Train with regularization parameters
# weight_decay=1e-3 penalizes large weights
reg_hist = train(reg_model, weight_decay=1e-3)

# Calculate test ROC-AUC for the regularized model
with torch.no_grad():
    reg_auc = roc_auc_score(yte_t.numpy(), torch.sigmoid(reg_model(Xte_t)).numpy())

# Print results
print('regularized test ROC-AUC: %.3f' % reg_auc)

# Plot and compare loss curves: baseline vs regularized
# TODO: Overlay validation curves to see how regularization reduces overfitting
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist['train'], label='baseline train')
axes[0].plot(hist['val'], label='baseline val')
axes[0].set_title('Baseline (No Regularization)')
axes[0].set_xlabel('epoch')
axes[0].set_ylabel('loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(reg_hist['train'], label='regularized train')
axes[1].plot(reg_hist['val'], label='regularized val')
axes[1].set_title('Regularized (Dropout + Weight Decay)')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('loss')
axes[1].legend()
axes[1].grid(True)
plt.show()

**Takeaway:** the four-line loop is the *same* for every PyTorch model — MLPs today,
transformers tomorrow. Regularization trades a little training fit for better generalization.